In [8]:
data = [
    {
        "prompt": "Nima uchun har kuni yetarli miqdorda suv ichish kerak?",
        "chosen": "Suv tana haroratini me'yorda saqlashga, oziq moddalarni tashishga va organizmning normal ishlashiga yordam beradi. Kun davomida muntazam suv ichish suvsizlanishning oldini oladi.",
        "rejected": "Suv ichishning sog'liq uchun foydasi yo'q. Faqat chanqagan paytda suv ichish kerak."
    },
    {
        "prompt": "Kattalar nima uchun har kecha yetarli uxlashi kerak?",
        "chosen": "Yetarli uyqu miya faoliyatini, xotirani va immun tizimini qo'llab-quvvatlaydi. Kattalarga odatda bir kechada 7–9 soat uyqu tavsiya etiladi.",
        "rejected": "Uyqu sog'liq uchun muhim emas. Kattalar xohlagancha kam uxlashi mumkin."
    },
    {
        "prompt": "Sog'lom nonushta qilishning qanday foydalari bor?",
        "chosen": "Sog'lom nonushta organizmga energiya beradi va diqqatni jamlashga yordam beradi. Tuxum, sabzavot, meva va suli kabi mahsulotlar yaxshi tanlov bo'lishi mumkin.",
        "rejected": "Nonushta qilish kerak emas. Ertalab hech narsa yemaslik har doim foydaliroq."
    },
    {
        "prompt": "Inson organizmiga oqsil nima uchun kerak?",
        "chosen": "Oqsil mushaklar, teri va boshqa to'qimalarning tiklanishi hamda o'sishi uchun zarur. Uni tuxum, baliq, go'sht, loviya va sut mahsulotlaridan olish mumkin.",
        "rejected": "Oqsil faqat sportchilar uchun kerak. Boshqa odamlar oqsil iste'mol qilmasa ham bo'ladi."
    },
    {
        "prompt": "Har kuni piyoda yurishning qanday foydasi bor?",
        "chosen": "Piyoda yurish yurak faoliyatini qo'llab-quvvatlaydi, kayfiyatni yaxshilaydi va jismoniy faollikni oshiradi. Imkoniyatga qarab, muntazam yurish foydali odat hisoblanadi.",
        "rejected": "Piyoda yurishning sog'liq uchun foydasi yo'q. Harakat qilmasdan dam olish yaxshiroq."
    },
    {
        "prompt": "D vitamini inson organizmida qanday vazifani bajaradi?",
        "chosen": "D vitamini organizmga kalsiyni o'zlashtirishga yordam beradi. U suyaklar va mushaklarning normal faoliyati uchun muhim.",
        "rejected": "D vitamini organizmga kerak emas. U suyaklar faoliyatiga ta'sir qilmaydi."
    },
    {
        "prompt": "Nima uchun meva va sabzavotlarni muntazam iste'mol qilish kerak?",
        "chosen": "Meva va sabzavotlar vitaminlar, minerallar va oziq tolalariga boy. Ular sog'lom ovqatlanishning muhim qismi hisoblanadi.",
        "rejected": "Meva va sabzavotlarda foydali moddalar yo'q. Ularning o'rniga faqat shirinlik yeyish mumkin."
    },
    {
        "prompt": "Stressni kamaytirish uchun qanday usullar foydali?",
        "chosen": "Chuqur nafas olish, piyoda yurish, yetarli uxlash va yaqin insonlar bilan suhbatlashish stressni kamaytirishga yordam berishi mumkin.",
        "rejected": "Stressni kamaytirish uchun doimo muammolarni ichda saqlash va hech kim bilan gaplashmaslik kerak."
    },
    {
        "prompt": "Mashq qilishdan oldin badan qizdirish nima uchun kerak?",
        "chosen": "Badan qizdirish mushaklarni jismoniy faoliyatga tayyorlaydi va harakatlarni bajarishni osonlashtiradi. Mashqdan oldin yengil qizish foydali.",
        "rejected": "Badan qizdirish kerak emas. Darhol og'ir mashqlarni boshlash mumkin."
    },
    {
        "prompt": "Oziq tolalari hazm qilish tizimi uchun nima sababdan muhim?",
        "chosen": "Oziq tolalari ichak faoliyatini qo'llab-quvvatlaydi va qabziyatning oldini olishga yordam beradi. Ular sabzavot, meva, dukkakli mahsulotlar va donlarda mavjud.",
        "rejected": "Oziq tolalari hazm qilishga yordam bermaydi. Ularni ovqatdan butunlay chiqarib tashlash kerak."
    }
]

ds = Dataset.from_list(data)

print("Training samples:", len(ds))
print("Dataset columns:", ds.column_names)


Training samples: 10
Dataset columns: ['prompt', 'chosen', 'rejected']


In [ ]:
import torch
import pandas as pd
from peft import PeftModel

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from peft import LoraConfig
from trl import RewardConfig, RewardTrainer


# 1. Modelni yuklash
model_id = "HuggingFaceTB/SmolLM2-360M"

tok = AutoTokenizer.from_pretrained(model_id)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

rm = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=1
)

rm.config.pad_token_id = tok.pad_token_id


# 2. LoRA va Training sozlamalari
peft_cfg = LoraConfig(
    task_type="SEQ_CLS",
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

args = RewardConfig(
    output_dir="smollm2-reward-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    learning_rate=1e-4,
    max_length=256,
    fp16=False,
    bf16=False,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)


# 3. Reward Modelni o'qitish
trainer = RewardTrainer(
    model=rm,
    args=args,
    processing_class=tok,
    train_dataset=ds,
    peft_config=peft_cfg
)

trainer.train()


# 4. Modelni saqlash
trainer.save_model("smollm2-reward-model")
tok.save_pretrained("smollm2-reward-model")

print("Reward Model saqlandi!")


# 5. Javobga RM ball berish
rm = trainer.model.eval()

def score(prompt, response):
    text = prompt + " " + response

    inputs = tok(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(rm.device)

    with torch.no_grad():
        result = rm(**inputs).logits[0, 0].item()

    return result


# 6. 10 ta namunani baholash
results = []

for i, item in enumerate(data, 1):
    chosen_score = score(
        item["prompt"],
        item["chosen"]
    )

    rejected_score = score(
        item["prompt"],
        item["rejected"]
    )

    results.append({
        "№": i,
        "Prompt": item["prompt"].replace("Savol: ", "").replace("\nJavob:", ""),
        "RM Chosen": round(chosen_score, 4),
        "RM Rejected": round(rejected_score, 4),
        "Farq": round(chosen_score - rejected_score, 4)
    })

df = pd.DataFrame(results)

print("\nREWARD MODEL NATIJALARI:")
display(df)

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 278.90it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-360M
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Filtering train >256 tokens: 100%|██████████| 10/10 [00:00<00:00, 706.61 examples/s]


Step,Training Loss
1,0.765625
2,0.593750
3,0.617188
4,0.585938
5,0.742188
6,0.593750
7,0.570312
8,0.574219
9,0.734375
10,0.703125


Reward Model saqlandi!

REWARD MODEL NATIJALARI:


,№,Prompt,RM Chosen,RM Rejected,Farq
0,1,Nima uchun har kuni yetarli miqdorda suv ichis...,1.2969,0.8750,0.4219
1,2,Kattalar nima uchun har kecha yetarli uxlashi ...,1.3672,0.7969,0.5703
2,3,Sog'lom nonushta qilishning qanday foydalari bor?,1.1953,1.1484,0.0469
3,4,Inson organizmiga oqsil nima uchun kerak?,1.0156,1.0469,-0.0312
4,5,Har kuni piyoda yurishning qanday foydasi bor?,1.3281,0.8945,0.4336
5,6,D vitamini inson organizmida qanday vazifani b...,1.2109,1.1641,0.0469
6,7,Nima uchun meva va sabzavotlarni muntazam iste...,1.1953,1.3281,-0.1328
7,8,Stressni kamaytirish uchun qanday usullar foyd...,1.3828,1.2969,0.0859
8,9,Mashq qilishdan oldin badan qizdirish nima uch...,1.0938,0.9414,0.1523
9,10,Oziq tolalari hazm qilish tizimi uchun nima sa...,1.1719,0.8789,0.2930
